# Notebook 04 — Model Training & Evaluation
Runs all models under protocols P1 (within-dataset), P2 (cross-population) and P3 (cross-dataset).

**Models:**
- Threshold baseline (Bourke et al., 2007)
- Random Forest
- SVM (RBF)
- XGBoost
- 1D-CNN

**Answers RQ2 (generalization gap) and RQ3 (cross-cohort transfer).**


In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/fall-detection-generalization'
    %pip install -q numpy pandas scipy scikit-learn xgboost torch tqdm imbalanced-learn
    sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    REPO_ROOT = str(pathlib.Path('..').resolve())

DATA_ROOT = REPO_ROOT + '/data/raw'
RESULTS   = REPO_ROOT + '/results'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

from src.data.sisfall import load_sisfall
from src.data.harmonize import TARGET_FS
from src.models.threshold import ThresholdDetector
from src.models.classical import build_classical_models
from src.models.deep import CNN1D, train_cnn, predict_cnn
from src.evaluation.protocols import run_p1_within, run_p2_cross_population, run_p3_cross_dataset, aggregate_results
from src.evaluation.metrics import compute_metrics, bootstrap_ci

RESULTS_DIR = Path(RESULTS)
RESULTS_DIR.mkdir(exist_ok=True)

## 1. Load pre-computed data (from Notebook 03)

In [ ]:
X_raw      = np.load(f'{RESULTS}/X_raw.npy')
y          = np.load(f'{RESULTS}/y.npy')
subjects   = np.load(f'{RESULTS}/subjects.npy', allow_pickle=True).tolist()
age_groups = np.load(f'{RESULTS}/age_groups.npy', allow_pickle=True).tolist()

df_feat = pd.read_parquet(f'{RESULTS}/features_filtered.parquet')
meta_cols = ['label', 'subject_id', 'age_group']
feat_cols = [c for c in df_feat.columns if c not in meta_cols]

X_feat = df_feat[feat_cols].values.astype(np.float32)
y_feat = df_feat['label'].values
subj_feat = df_feat['subject_id'].tolist()
grp_feat  = df_feat['age_group'].tolist()

print(f'Raw windows: {X_raw.shape}')
print(f'Feature matrix: {X_feat.shape}')

## 2. Protocol P1 — Within-dataset, subject-independent (grouped 5-fold CV)

In [ ]:
classical_models = build_classical_models(random_state=42)
all_results = []

for name, model in tqdm(classical_models.items(), desc='P1 classical'):
    df_p1 = run_p1_within(model, X_feat, y_feat, subj_feat,
                           n_splits=5, model_name=name)
    all_results.append(df_p1)
    mean_sens = df_p1['sensitivity'].mean()
    mean_f1   = df_p1['f1'].mean()
    print(f'  {name:20s}  sensitivity={mean_sens:.3f}  F1={mean_f1:.3f}')

In [ ]:
# 1D-CNN — P1
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
import torch

gkf = GroupKFold(n_splits=5)
cnn_p1_rows = []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_raw, y, np.array(subjects))):
    model_cnn = CNN1D(n_channels=X_raw.shape[2], window_len=X_raw.shape[1], n_classes=2)
    train_cnn(model_cnn, X_raw[tr_idx], y[tr_idx], epochs=20, batch_size=64)
    y_pred = predict_cnn(model_cnn, X_raw[te_idx])
    metrics = compute_metrics(y[te_idx], y_pred)
    metrics.update({'protocol': 'P1', 'fold': fold, 'model': '1d_cnn',
                    'n_train': len(tr_idx), 'n_test': len(te_idx)})
    cnn_p1_rows.append(metrics)

df_cnn_p1 = pd.DataFrame(cnn_p1_rows)
all_results.append(df_cnn_p1)
print(f'CNN P1  sensitivity={df_cnn_p1["sensitivity"].mean():.3f}  F1={df_cnn_p1["f1"].mean():.3f}')

## 3. Protocol P2 — Cross-population (young → elderly)

In [ ]:
p2_rows = []

for name, model in classical_models.items():
    for direction in ['young_to_elderly', 'elderly_to_young']:
        try:
            row = run_p2_cross_population(model, X_feat, y_feat, grp_feat,
                                           model_name=name, direction=direction)
            p2_rows.append(row)
        except ValueError as e:
            print(f'  Skipping {name} {direction}: {e}')

df_p2 = pd.DataFrame(p2_rows)
all_results.append(df_p2)
print(df_p2[['model', 'direction', 'sensitivity', 'specificity', 'f1']].to_string(index=False))

## 4. Threshold baseline — P1 and P2 (trial-level)

In [ ]:
trials = load_sisfall(Path(DATA_ROOT) / 'sisfall')
y_all  = np.array([int(t.is_fall) for t in trials])
grps_all = [t.age_group for t in trials]

det = ThresholdDetector(fs=200)  # SisFall is 200 Hz

# P1 — all data
y_pred_thr = det.predict(trials)
thr_metrics = compute_metrics(y_all, y_pred_thr)
thr_metrics.update({'protocol': 'P1_trial', 'model': 'threshold'})
print('Threshold P1 (all subjects):')
for k, v in thr_metrics.items():
    if k in ('sensitivity', 'specificity', 'f1', 'tp', 'fp', 'fn', 'tn'):
        print(f'  {k}: {v}')

# P2 — young trained, test elderly
best_imp, best_ina, best_f1 = det.tune_threshold(
    [t for t in trials if t.age_group == 'young' and t.is_fall],
)
print(f'\nThreshold tuned on young: impact={best_imp:.2f}g  inactivity={best_ina:.2f}g  F1={best_f1:.3f}')

elderly_trials = [t for t in trials if t.age_group == 'elderly']
y_eld = np.array([int(t.is_fall) for t in elderly_trials])
y_pred_eld = det.predict(elderly_trials)
thr_elderly = compute_metrics(y_eld, y_pred_eld)
thr_elderly.update({'protocol': 'P2', 'direction': 'young_to_elderly', 'model': 'threshold'})
print('\nThreshold P2 (young-tuned on elderly):  sensitivity=', thr_elderly['sensitivity'])

## 5. Aggregate and visualize results

In [ ]:
df_all = pd.concat(all_results, ignore_index=True)
df_all.to_csv(f'{RESULTS}/all_results.csv', index=False)

# P1 summary
df_p1_all = df_all[df_all['protocol'] == 'P1']
p1_summary = df_p1_all.groupby('model')[['sensitivity', 'specificity', 'f1', 'balanced_acc']].agg(['mean', 'std'])
print('=== P1 Summary ===')
print(p1_summary.round(3).to_string())

In [ ]:
# P1 vs P2 — generalization gap visualization
metric = 'sensitivity'
fig, ax = plt.subplots(figsize=(10, 5))

p1_means = df_p1_all.groupby('model')[metric].mean().rename('P1_within')
p2_young_to_eld = df_p2[df_p2['direction'] == 'young_to_elderly'].set_index('model')[metric].rename('P2_young→elderly')

gap_df = pd.concat([p1_means, p2_young_to_eld], axis=1).dropna()
gap_df.plot(kind='bar', ax=ax, rot=0, color=['#4c72b0', '#dd8452'])

ax.set_title(f'{metric.capitalize()} — P1 (within) vs P2 (young→elderly)')
ax.set_ylabel(metric)
ax.set_ylim(0, 1)
ax.axhline(0.95, ls='--', color='grey', alpha=0.5, label='target 0.95')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS}/rq2_generalization_gap.png', bbox_inches='tight')
plt.show()

print('\nGeneralization gap (P1 → P2 drop):')
print((gap_df['P1_within'] - gap_df['P2_young→elderly']).round(3).rename('gap').to_string())

In [ ]:
# Confusion matrices for best classical model and 1D-CNN under P2
from sklearn.metrics import ConfusionMatrixDisplay

models_to_show = ['random_forest', '1d_cnn']
fig, axes = plt.subplots(1, len(models_to_show), figsize=(10, 4))

for ax, mname in zip(axes, models_to_show):
    row = df_p2[df_p2['model'] == mname]
    if row.empty:
        ax.set_visible(False)
        continue
    r = row.iloc[0]
    cm = np.array([[r['tn'], r['fp']], [r['fn'], r['tp']]])
    disp = ConfusionMatrixDisplay(cm, display_labels=['ADL', 'Fall'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f'{mname} — P2 young→elderly')

plt.tight_layout()
plt.savefig(f'{RESULTS}/p2_confusion_matrices.png', bbox_inches='tight')
plt.show()

## 6. Protocol P3 — Cross-dataset (requires Laavanya dataset)

In [ ]:
LAV_FEAT = Path(RESULTS) / 'laavanya_features_filtered.parquet'

if LAV_FEAT.exists():
    df_lav = pd.read_parquet(LAV_FEAT)
    X_lav = df_lav[feat_cols].values.astype(np.float32)
    y_lav = df_lav['label'].values

    p3_rows = []
    for name, model in classical_models.items():
        # SisFall → Laavanya
        row = run_p3_cross_dataset(model, X_feat, y_feat, X_lav, y_lav,
                                    model_name=name,
                                    train_dataset='sisfall',
                                    test_dataset='laavanya')
        p3_rows.append(row)
        # Laavanya → SisFall elderly
        elderly_mask = np.array(grp_feat) == 'elderly'
        row2 = run_p3_cross_dataset(model, X_lav, y_lav,
                                     X_feat[elderly_mask], y_feat[elderly_mask],
                                     model_name=name,
                                     train_dataset='laavanya',
                                     test_dataset='sisfall_elderly')
        p3_rows.append(row2)

    df_p3 = pd.DataFrame(p3_rows)
    print(df_p3[['model', 'train_dataset', 'test_dataset', 'sensitivity', 'f1']].to_string(index=False))
    df_p3.to_csv(f'{RESULTS}/p3_results.csv', index=False)
else:
    print('Laavanya features not found — run this cell after completing Notebook 03 for the Laavanya dataset.')